# 05 — Isolation Forest Anomaly Detection

Unsupervised anomaly scoring over the time-window features. **Evaluation design matches notebook 04:** the forest is fit on the first 70% of transactions (past) and evaluated on the final 20% (future).

Key decisions:
- **Scaling kept, but fit on train only** — IsolationForest is not scale-invariant (random split thresholds depend on within-feature distances; verified empirically). `log1p` tames heavy-tailed amount/count columns first.
- **Score normalization uses train-window min/max only** — full-data stats would leak future extremes into past scores.
- Output `anomaly_score` feeds the hybrid model in notebook 06; the model bundle (`model + scaler + transforms + score stats`) is saved for the FastAPI service.

In [ ]:
# CELL 1 - Load + sort chronologically (same design as 04)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score

df = pd.read_csv("../data/processed/time_window_features.csv")
df["TransactionDT"] = pd.to_datetime(df["TransactionDT"])

# Chronological order: anomaly model is FIT on the past, SCORED on
# the future - identical evaluation design as notebook 04, so the
# models stay directly comparable.
df = df.sort_values("TransactionDT").reset_index(drop=True)

os.makedirs("../models", exist_ok=True)
os.makedirs("../reports", exist_ok=True)
print(f"Loaded: {df.shape}")

In [ ]:
# CELL 2 - Anomaly feature set
if "velocity_risk" not in df.columns:   # same fallback as 04
    df["velocity_risk"] = (
        df["txn_count_1h"] / df["txn_count_24h"].clip(lower=1)
    ).clip(0, 1)

anomaly_features = [
    "TransactionAmt",
    "txn_count_1h", "txn_count_24h", "txn_count_7d", "velocity_risk",
    "avg_amt_24h", "amount_dev_24h", "amount_zscore_24h",
    "is_night_txn", "no_prior_24h",
]
missing = [f for f in anomaly_features if f not in df.columns]
assert not missing, f"Missing columns: {missing} - re-run notebooks 01-03 first"

In [ ]:
# CELL 3 - Time-based fit/test windows (match notebook 04 exactly)
n = len(df)
train_end = int(n * 0.70)      # fit window: first 70%
test_start = int(n * 0.80)     # evaluation: last 20% (same as 04's test)

test_mask = df.index >= test_start
print(f"fit window:      {train_end:,} rows (past)")
print(f"evaluate window: {test_mask.sum():,} rows (future)")

In [ ]:
# CELL 4 - Transform + train Isolation Forest on PAST data only
# IsolationForest is NOT scale-invariant: random split thresholds are
# drawn between a feature's observed min/max, so within-feature
# distances change the learned rankings (verified empirically:
# Spearman ~0.50 between raw and scaled scores on identical data).
# -> log1p the heavy-tailed money/count columns, then scale with a
#    scaler FIT ON THE TRAIN WINDOW ONLY (no future leakage).
from sklearn.preprocessing import StandardScaler

log_cols = ["TransactionAmt", "txn_count_1h", "txn_count_24h",
            "txn_count_7d", "avg_amt_24h"]

X_all = df[anomaly_features].copy()
for c in log_cols:
    X_all[c] = np.log1p(X_all[c])

X_train = X_all.iloc[:train_end]

scaler = StandardScaler().fit(X_train)          # fit on past only
X_train_s = scaler.transform(X_train)
X_all_s = scaler.transform(X_all)

iso = IsolationForest(
    n_estimators=200,
    contamination=0.03,     # ~ expected anomalous share in fit window
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_train_s)
print(f"Fitted on {len(X_train):,} past transactions.")

In [ ]:
# CELL 5 - Score all rows; normalize with TRAIN-window stats only
raw = -iso.score_samples(X_all_s)     # higher = more anomalous

train_raw = raw[:train_end]
lo, hi = train_raw.min(), train_raw.max()
df["anomaly_score"] = np.clip((raw - lo) / (hi - lo), 0, 1)

In [ ]:
# CELL 6 - Evaluate on the held-out FUTURE window
yt = df.loc[test_mask, "isFraud"].to_numpy()
sc = df.loc[test_mask, "anomaly_score"].to_numpy()
base = yt.mean()

top_k = max(1, int(0.03 * len(sc)))          # alert budget: top 3%
top_idx = np.argsort(sc)[-top_k:]
lift = yt[top_idx].mean() / base

print("TEST (future window) - anomaly_score performance:")
print(f"  ROC-AUC: {roc_auc_score(yt, sc):.4f}")
print(f"  PR-AUC:  {average_precision_score(yt, sc):.4f}   (baseline = {base:.4f})")
print(f"  fraud rate in top 3% scores: {yt[top_idx].mean():.2%}  ({lift:.1f}x lift)\n")
print(df.loc[test_mask].groupby("isFraud")["anomaly_score"].mean().round(4))

In [ ]:
# CELL 7 - Score distributions (test window), saved
test_df = df.loc[test_mask]
plt.figure(figsize=(8, 4.5))
plt.hist(test_df.loc[test_df.isFraud == 0, "anomaly_score"],
         bins=60, alpha=0.6, density=True, label="Non-fraud")
plt.hist(test_df.loc[test_df.isFraud == 1, "anomaly_score"],
         bins=60, alpha=0.6, density=True, label="Fraud")
plt.xlabel("Anomaly score (0-1, train-normalized)")
plt.ylabel("Density")
plt.title("Isolation Forest Scores: Fraud vs Non-Fraud (future window)")
plt.legend()
plt.savefig("../reports/iforest_score_distribution.png", dpi=150,
            bbox_inches="tight")   # BEFORE show()
plt.show()

In [ ]:
# CELL 8 - Save enriched CSV + model bundle (for the API)
import joblib

df.to_csv("../data/processed/fraud_features_with_anomaly_iforest.csv",
          index=False)

# Bundle everything needed to score a LIVE transaction consistently:
joblib.dump(
    {"model": iso, "scaler": scaler,
     "log_cols": log_cols, "features": anomaly_features,
     "score_min": float(lo), "score_max": float(hi)},
    "../models/iforest_anomaly_model.pkl",
)
print(f"Saved CSV: {df.shape}")
print("Saved model bundle -> ../models/iforest_anomaly_model.pkl")